# Data batch processing

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import os
from scipy.stats import entropy
from concurrent.futures import ThreadPoolExecutor, as_completed

/Users/jasonc/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/jasonc/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/jasonc/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
consumer_df = pd.read_parquet("/Users/jasonc/Desktop/DSC_291/cashflow/consumer_data.parquet")

In [3]:
# FFT Feature Function
def extract_fft_features(ts, detrend=True, low_freq_cut=0.05, high_freq_cut=0.25):
    ts = ts.asfreq('W-MON').fillna(0)
    if detrend:
        ts = ts - ts.mean()

    fft_vals = np.fft.fft(ts)
    fft_freqs = np.fft.fftfreq(len(ts), d=1)
    pos_mask = fft_freqs > 0
    freqs = fft_freqs[pos_mask]
    mags = np.abs(fft_vals)[pos_mask]
    power = mags**2
    power_norm = power / power.sum()

    if len(power) == 0:
        return None

    dominant_idx = np.argmax(power)
    dominant_freq = freqs[dominant_idx]
    dominant_power = power[dominant_idx]
    spectral_entropy = entropy(power_norm)
    low_power = power[freqs < low_freq_cut].sum()
    high_power = power[freqs > high_freq_cut].sum()
    power_ratio = low_power / (high_power + 1e-6)

    return {
        "dominant_freq": dominant_freq,
        "dominant_power": dominant_power,
        "spectral_entropy": spectral_entropy,
        "low_freq_power": low_power,
        "high_freq_power": high_power,
        "power_ratio": power_ratio
    }

In [4]:
def weekly_trimmed(ts):
    ts = ts.groupby(ts.index).sum()  # collapse duplicates
    ts = ts.asfreq('D').fillna(0)
    if ts.empty:
        return pd.Series(dtype=float)

    first_monday = ts.index[0] + pd.DateOffset(days=(7 - ts.index[0].weekday()) % 7)
    last_sunday = ts.index[-1] - pd.DateOffset(days=(ts.index[-1].weekday() + 1) % 7)
    ts_trimmed = ts[(ts.index >= first_monday) & (ts.index <= last_sunday)]

    return ts_trimmed.resample('W-MON').sum()

In [5]:
# Consumer Processor
def process_consumer_file(file_path):
    try:
        consumer_id = os.path.basename(file_path).replace(".parquet", "")
        df = pd.read_parquet(file_path)

        # Safely convert to datetime
        df['posted_date'] = pd.to_datetime(df['posted_date'], errors='coerce')
        df = df.dropna(subset=['posted_date'])
        df = df.sort_values('posted_date')

        if df.empty:
            raise ValueError("No valid posted_date after cleaning")

        # Resample to daily and fill forward
        ts = (
            df.set_index('posted_date')
            .resample('D')['amount']
            .sum()
            .ffill()
        )

        if ts.empty or ts.sum() == 0 or ts.std() == 0:
            raise ValueError("Time series is empty or flat")

        features = extract_fft_features(ts)
        if features is None:
            raise ValueError("FFT failed due to empty power spectrum")

        features['masked_consumer_id'] = consumer_id
        print(f"✓ Processed {consumer_id}")
        return features

    except Exception as e:
        print(f"✗ Failed {file_path}: {e}")
        return None

In [6]:
def process_consumer_file_by_category(file_path):
    try:
        consumer_id = os.path.basename(file_path).replace(".parquet", "")
        df = pd.read_parquet(file_path)

        df['posted_date'] = pd.to_datetime(df['posted_date'], errors='coerce')
        df = df.dropna(subset=['posted_date'])
        df = df.sort_values('posted_date')

        if df.empty:
            raise ValueError("No valid posted_date after cleaning")

        category_features = {}

        for cat, sub_df in df.groupby('category'):
            ts = sub_df.set_index('posted_date')['amount']
            ts_weekly = weekly_trimmed(ts)

            if ts_weekly.empty or ts_weekly.sum() == 0 or ts_weekly.std() == 0 or len(ts_weekly) < 6:
                continue

            features = extract_fft_features(ts_weekly)
            if features is None:
                continue

            for key, val in features.items():
                category_features[f"{key}_cat{int(cat)}"] = val

        if not category_features:
            raise ValueError("All categories failed")

        category_features['masked_consumer_id'] = consumer_id
        print(f"✔ Processed {consumer_id}")
        return category_features

    except Exception as e:
        print(f"✖ Failed {file_path}: {e}")
        return None

In [7]:
def batch_process(tx_dir):
    file_paths = [os.path.join(tx_dir, f) for f in os.listdir(tx_dir) if f.endswith(".parquet")]

    all_features = []
    with ThreadPoolExecutor() as executor:
        futures = {executor.submit(process_consumer_file_by_category, path): path for path in file_paths}
        for future in as_completed(futures):
            result = future.result()
            if result:
                all_features.append(result)

    df_result = pd.DataFrame(all_features)
    print(f"\n✅ Finished processing {len(df_result)} consumers.")
    return df_result

In [8]:
if __name__ == "__main__":
    TX_DIR = "/Users/jasonc/Desktop/DSC_291/split_transactions"
    output_df = batch_process(TX_DIR)

    # Optional: Save
    output_df.to_csv("/Users/jasonc/Desktop/DSC_291/processed_data.csv")

✔ Processed C02100394
✔ Processed C02104921
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03104720.parquet: All categories failed
✔ Processed C03102314
✔ Processed C03100574
✔ Processed C02103967
✔ Processed C04103556
✔ Processed C02103186
✔ Processed C04101336
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03101106.parquet: No valid posted_date after cleaning
✔ Processed C03104658
✔ Processed C03103766
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03104648.parquet: All categories failed
✔ Processed C04102124
✔ Processed C01102837
✔ Processed C04104468
✔ Processed C02104859
✔ Processed C04104510
✔ Processed C03104730
✔ Processed C04101326
✔ Processed C04104500
✔ Processed C02103977
✔ Processed C02104931
✔ Processed C02103196
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03103776.parquet: No valid posted_date after cleaning
✔ Processed C02104849
✔ Processed C04103546
✔ Processed C03104994
✔ Processed C04100754
✔ Processed C04104478
✔ Pr

/var/folders/vb/p9mktqwx0rg59m22m1fdfbk80000gn/T/ipykernel_22906/683539865.py:13: RuntimeWarning: invalid value encountered in divide
  power_norm = power / power.sum()


✔ Processed C03100673
✔ Processed C02100093
✔ Processed C02103291
✔ Processed C04102233
✔ Processed C02100862
✔ Processed C04103739
✔ Processed C04100453
✔ Processed C04103651
✔ Processed C04101159
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03103509.parquet: No valid posted_date after cleaning
✔ Processed C02100083
✔ Processed C02101589
✔ Processed C03100663✔ Processed C04103641

✔ Processed C03102003
✔ Processed C01101932
✔ Processed C03104437
✔ Processed C04101021
✔ Processed C04104607
✔ Processed C01104829
✔ Processed C02104492
✔ Processed C03104272
✔ Processed C03101369
✔ Processed C03103234
✔ Processed C01101796
✔ Processed C01103917
✔ Processed C04100216
✔ Processed C04101664
✔ Processed C02102847
✔ Processed C03101454
✔ Processed C04102476
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03103224.parquet: All categories failed
✔ Processed C04103004
✔ Processed C01104839
✔ Processed C04102466
✔ Processed C01102584
✔ Processed C02102857
✔ Processed C03104

/var/folders/vb/p9mktqwx0rg59m22m1fdfbk80000gn/T/ipykernel_22906/683539865.py:13: RuntimeWarning: invalid value encountered in divide
  power_norm = power / power.sum()


✔ Processed C03100734
✔ Processed C04102364
✔ Processed C03102144
✔ Processed C03102154
✔ Processed C04104740
✔ Processed C02104390
✔ Processed C03100724
✔ Processed C02100925
✔ Processed C04101987
✔ Processed C04101166
✔ Processed C03104408
✔ Processed C01101094
✔ Processed C04100514
✔ Processed C03104570
✔ Processed C04101723
✔ Processed C04100229
✔ Processed C02102878
✔ Processed C04104105
✔ Processed C02103593
✔ Processed C03100161
✔ Processed C03103536
✔ Processed C01103928
✔ Processed C04103143
✔ Processed C03102701
✔ Processed C03100019
✔ Processed C03100980
✔ Processed C01104816
✖ Failed /Users/jasonc/Desktop/DSC_291/split_transactions/C03102711.parquet: No valid posted_date after cleaning
✔ Processed C02102900
✔ Processed C04103153
✔ Processed C04102531
✔ Processed C04100351
✔ Processed C01104806
✔ Processed C02100791
✔ Processed C03101513
✔ Processed C02102868
✔ Processed C02102910✔ Processed C01103840

✔ Processed C03101503
✔ Processed C04104115
✔ Processed C03102679
✔ Proce

/var/folders/vb/p9mktqwx0rg59m22m1fdfbk80000gn/T/ipykernel_22906/683539865.py:13: RuntimeWarning: invalid value encountered in divide
  power_norm = power / power.sum()


✔ Processed C04102787
✔ Processed C01104339
✔ Processed C01102675
✔ Processed C04100906
✔ Processed C02101145
✔ Processed C03101098
✔ Processed C01101467✔ Processed C03104183

✔ Processed C03101879
✔ Processed C02102357
✔ Processed C02103725
✔ Processed C03103934
✔ Processed C04100916
✔ Processed C01104241
✔ Processed C04101585
✔ Processed C02103735
✔ Processed C02100527
✔ Processed C03104193
✔ Processed C01100005
✔ Processed C04102797
✔ Processed C01102665
✔ Processed C03104962
✔ Processed C01104251
✔ Processed C02101155
✔ Processed C03104972
✔ Processed C02100651
✔ Processed C02104773
✔ Processed C01103217
✔ Processed C01103980
✔ Processed C01101679
✔ Processed C02102031
✔ Processed C04103093
✔ Processed C01104127
✔ Processed C01103019
✔ Processed C03100928
✔ Processed C02103443
✔ Processed C04102599
✔ Processed C01100373
✔ Processed C04100281
✔ Processed C02102149
✔ Processed C01102513
✔ Processed C02104405
✔ Processed C02101223
✔ Processed C02100729
✔ Processed C01101701
✔ Processe

/var/folders/vb/p9mktqwx0rg59m22m1fdfbk80000gn/T/ipykernel_22906/683539865.py:13: RuntimeWarning: invalid value encountered in divide
  power_norm = power / power.sum()


✔ Processed C01103994
✔ Processed C04104958
✔ Processed C01102507
✔ Processed C02103721
✔ Processed C04103087
✔ Processed C04103289
✔ Processed C02104767
✔ Processed C03104389
✔ Processed C01100169
✔ Processed C01104245
✔ Processed C02101141
✔ Processed C01103203
✔ Processed C02103659
✔ Processed C02100533
✔ Processed C04102783
✔ Processed C02101039
✔ Processed C01102709
✔ Processed C01102719
✔ Processed C04103299
✔ Processed C03104197
✔ Processed C01100179
✔ Processed C04100902
✔ Processed C02103731
✔ Processed C01103213
✔ Processed C01101473
✔ Processed C02101151
✔ Processed C02100523
✔ Processed C02101029
✔ Processed C03104966
✔ Processed C01100998
✔ Processed C04100912
✔ Processed C01102661
✔ Processed C03103920
✔ Processed C02102343
✔ Processed C03104976
✔ Processed C04101581
✔ Processed C02103649
✔ Processed C03103930
✔ Processed C02103174
✔ Processed C01104410
✔ Processed C03103794
✔ Processed C02104132
✔ Processed C03103848
✔ Processed C01100001
✔ Processed C02101714
✔ Processe

In [13]:
# Merge the FPF_TARGET column into the features DataFrame
output_df = output_df.merge(
    consumer_df[['masked_consumer_id', 'FPF_TARGET']],
    on='masked_consumer_id',
    how='left'
)

In [36]:
# Sort by FPF_TARGET and set masked_consumer_id as the index
output_df_sorted = output_df.sort_values(by='FPF_TARGET')
output_df_sorted = output_df_sorted[sorted(output_df_sorted.columns)]

In [37]:
output_df_sorted = output_df_sorted.fillna(0)
output_df_sorted = output_df_sorted.set_index('masked_consumer_id')
output_df_sorted.to_csv("/Users/jasonc/Desktop/DSC_291/processed_data.csv")

In [38]:
output_df_sorted

,FPF_TARGET,dominant_freq_cat0,dominant_freq_cat1,dominant_freq_cat10,dominant_freq_cat11,dominant_freq_cat12,dominant_freq_cat13,dominant_freq_cat14,dominant_freq_cat15,dominant_freq_cat16,...,spectral_entropy_cat32,spectral_entropy_cat33,spectral_entropy_cat34,spectral_entropy_cat35,spectral_entropy_cat4,spectral_entropy_cat5,spectral_entropy_cat6,spectral_entropy_cat7,spectral_entropy_cat8,spectral_entropy_cat9
masked_consumer_id,,,,,,,,,,,,,,,,,,,,,
C02100394,0.0,0.000000,0.240000,0.000000,0.000000,0.250000,0.315789,0.461538,0.480000,0.240000,...,0.0,0.0,0.0,0.000000,1.988392,0.000000,1.098612,1.684803,0.000000,0.0
C01102904,0.0,0.000000,0.078431,0.000000,0.000000,0.229167,0.229167,0.023810,0.230769,0.038462,...,0.0,0.0,0.0,0.000000,2.944439,0.000000,2.890372,1.925229,0.000000,0.0
C03100447,0.0,0.078947,0.226667,0.466667,0.230769,0.453333,0.208333,0.213333,0.472973,0.013333,...,0.0,0.0,0.0,3.135449,0.000000,0.000000,3.241178,3.210977,0.000000,0.0
C04103703,0.0,0.000000,0.100000,0.000000,0.000000,0.300000,0.000000,0.100000,0.300000,0.100000,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
C04104745,0.0,0.326531,0.423077,0.000000,0.000000,0.220000,0.000000,0.235294,0.431373,0.183673,...,0.0,0.0,0.0,0.000000,2.766986,0.000000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C03102986,1.0,0.071429,0.012987,0.000000,0.000000,0.013889,0.388889,0.020408,0.117647,0.061224,...,0.0,0.0,0.0,0.000000,3.531708,0.000000,2.096233,0.000000,0.000000,0.0
C02100699,1.0,0.040000,0.461538,0.000000,0.000000,0.000000,0.000000,0.038462,0.391304,0.230769,...,0.0,0.0,0.0,0.000000,1.945910,2.378646,0.000000,0.000000,0.000000,0.0
C02102960,1.0,0.360000,0.440000,0.478261,0.000000,0.000000,0.000000,0.083333,0.480000,0.440000,...,0.0,0.0,0.0,0.000000,2.349013,0.000000,0.000000,0.000000,0.000000,0.0
